In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_8.py ──
"""
Shared infrastructure for MLFP02 Exercise 8 — FeatureStore + Feature Engineering.

Contains: HDB resale data loading, FeatureStore / ExperimentTracker setup
through kailash-ml, and OLS-from-scratch helpers reused across the four
R10 technique files:

    01_feature_schema.py        — FeatureSchema v1 + validation
    02_point_in_time.py         — Leakage prevention + temporal correctness
    03_rolling_features.py      — FeatureSchema v2 + group_by_dynamic
    04_modeling_with_features.py — Regression + hypothesis tests + Bayes

Technique-specific logic (schema construction, rolling window design,
coefficient interpretation) belongs in the per-technique files. This
module only owns infrastructure and reusable numeric helpers.
"""

from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp02_ex8"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_STORE_URL = "sqlite:///mlfp02_ex8_features.db"
FEATURE_TABLE_PREFIX = "kml_feat_"
EXPERIMENT_NAME = "mlfp02_ex8_hdb_features"


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flats (data.gov.sg)
# ════════════════════════════════════════════════════════════════════════


def load_hdb_resale() -> pl.DataFrame:
    """Load HDB resale transactions with a parsed transaction_date column.

    The raw file stores `month` as "YYYY-MM"; we convert it to a polars
    Date so every downstream technique can sort, filter, and roll on a
    real temporal axis without string parsing.
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")
    hdb = hdb.with_columns(
        pl.col("month").str.to_date("%Y-%m").alias("transaction_date")
    )
    return hdb


# ════════════════════════════════════════════════════════════════════════
# FEATURE STORE + EXPERIMENT TRACKER — kailash-ml wiring
# ════════════════════════════════════════════════════════════════════════


async def setup_feature_store() -> tuple[Any, Any, Any, bool]:
    """Create (conn, FeatureStore, ExperimentTracker, has_backend) for kailash-ml 1.1.1.

    Returns ``has_backend=False`` if the infrastructure is unavailable.
    Callers handle the degraded path by running the Polars-only versions
    of each operation.

    Note: the first tuple element is now a ``ConnectionManager`` rather than
    the old ``StoreFactory`` — kailash-ml's ExperimentTracker no longer
    accepts a positional store object; it constructs its own through the
    ``store_url`` factory. We still return a ConnectionManager so FeatureStore
    has the connection it needs.
    """
    try:
        from kailash.db import ConnectionManager
        from kailash_ml import ExperimentTracker, FeatureStore

        conn = ConnectionManager(FEATURE_STORE_URL)
        await conn.initialize()
        fs = FeatureStore(conn, table_prefix=FEATURE_TABLE_PREFIX)
        tracker = await ExperimentTracker.create(store_url=FEATURE_STORE_URL)
        return conn, fs, tracker, True
    except Exception as exc:  # noqa: BLE001 — degrade gracefully
        print(
            f"  [warn] FeatureStore backend unavailable "
            f"({type(exc).__name__}: {exc})"
        )
        return None, None, None, False


# ════════════════════════════════════════════════════════════════════════
# FEATURE COMPUTATION — v1 (basic property) and v2 (rolling market)
# ════════════════════════════════════════════════════════════════════════


def compute_v1_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compute version-1 HDB property features from raw transactions.

    Produces: storey_midpoint, price_per_sqm, remaining_lease_years,
    transaction_id (row index). These are the base features v2 extends.
    """
    return df.with_columns(
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2
        ).alias("storey_midpoint"),
        (pl.col("resale_price") / pl.col("floor_area_sqm")).alias("price_per_sqm"),
        (99 - (pl.col("transaction_date").dt.year() - pl.col("lease_commence_date")))
        .cast(pl.Float64)
        .alias("remaining_lease_years"),
    ).with_row_index("transaction_id")


def compute_v2_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compute v2 features = v1 + rolling town-level market context.

    Uses polars ``group_by_dynamic`` on ``transaction_date`` bucketed by
    month, then a 6-month rolling window per town. The first six months
    per town have nulls (warm-up period) — callers must ``drop_nulls``
    before modelling.
    """
    result = compute_v1_features(df).sort("transaction_date")

    town_stats = (
        result.group_by_dynamic("transaction_date", every="1mo", group_by="town")
        .agg(
            pl.col("resale_price").median().alias("monthly_median"),
            pl.col("resale_price").count().alias("monthly_volume"),
        )
        .sort("town", "transaction_date")
    )

    town_stats = town_stats.with_columns(
        pl.col("monthly_median")
        .rolling_mean(window_size=6)
        .over("town")
        .alias("town_median_price"),
        pl.col("monthly_volume")
        .rolling_sum(window_size=6)
        .over("town")
        .alias("town_transaction_volume"),
        (
            (pl.col("monthly_median") - pl.col("monthly_median").shift(6).over("town"))
            / pl.col("monthly_median").shift(6).over("town")
            * 100
        ).alias("town_price_trend"),
    )

    result = result.join(
        town_stats.select(
            "town",
            "transaction_date",
            "town_median_price",
            "town_transaction_volume",
            "town_price_trend",
        ),
        on=["town", "transaction_date"],
        how="left",
    )
    return result


# ════════════════════════════════════════════════════════════════════════
# POINT-IN-TIME RETRIEVAL HELPERS
# ════════════════════════════════════════════════════════════════════════


def as_of(
    df: pl.DataFrame, cutoff: datetime, date_col: str = "transaction_date"
) -> pl.DataFrame:
    """Return rows strictly before ``cutoff`` — the Polars-only PIT path.

    When FeatureStore is unavailable, every technique falls back to this
    helper so the leakage-prevention lesson still runs end-to-end.
    """
    return df.filter(pl.col(date_col) < pl.lit(cutoff.date()))


# ════════════════════════════════════════════════════════════════════════
# FROM-SCRATCH OLS HELPERS — reused across techniques 3 and 4
# ════════════════════════════════════════════════════════════════════════

FEATURE_LIST: list[str] = [
    "floor_area_sqm",
    "storey_midpoint",
    "remaining_lease_years",
    "town_median_price",
    "town_price_trend",
]


def prepare_design_matrix(
    df: pl.DataFrame,
    feature_list: list[str] = FEATURE_LIST,
    target: str = "resale_price",
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Drop nulls, build ``[1, X]`` design matrix, return ``(X, y, names)``."""
    model_data = df.drop_nulls(subset=[*feature_list, target])
    X_raw = model_data.select(feature_list).to_numpy().astype(np.float64)
    y = model_data[target].to_numpy().astype(np.float64)
    X = np.column_stack([np.ones(len(y)), X_raw])
    names = ["intercept", *feature_list]
    return X, y, names


def fit_ols(X: np.ndarray, y: np.ndarray) -> dict[str, Any]:
    """Fit OLS from scratch and return a dict with betas, SEs, t, p, R²."""
    from scipy import stats as sp_stats  # local import — optional at module load

    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    y_hat = X @ beta
    resid = y - y_hat

    ssr = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - ssr / sst
    adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - k)
    rmse = float(np.sqrt(ssr / n))

    sigma_sq = ssr / (n - k)
    xtx_inv = np.linalg.inv(X.T @ X)
    se = np.sqrt(sigma_sq * np.diag(xtx_inv))
    t_stat = beta / se
    p_val = 2.0 * (1.0 - sp_stats.t.cdf(np.abs(t_stat), df=n - k))

    sse = float(np.sum((y_hat - y.mean()) ** 2))
    f_stat = (sse / (k - 1)) / (ssr / (n - k))
    f_p = 1.0 - sp_stats.f.cdf(f_stat, dfn=k - 1, dfd=n - k)

    return {
        "n": n,
        "k": k,
        "beta": beta,
        "se": se,
        "t": t_stat,
        "p": p_val,
        "y_hat": y_hat,
        "resid": resid,
        "r2": float(r2),
        "adj_r2": float(adj_r2),
        "rmse": rmse,
        "f_stat": float(f_stat),
        "f_p": float(f_p),
    }


def normal_normal_posterior(
    beta_hat: float,
    se_hat: float,
    mu_prior: float = 0.0,
    sigma_prior: float = 10_000.0,
) -> dict[str, float]:
    """Normal-Normal conjugate posterior for a single OLS coefficient."""
    prec_prior = 1.0 / sigma_prior**2
    prec_data = 1.0 / se_hat**2
    prec_post = prec_prior + prec_data
    mu_post = (mu_prior * prec_prior + beta_hat * prec_data) / prec_post
    sigma_post = float(np.sqrt(1.0 / prec_post))
    return {
        "mu_post": float(mu_post),
        "sigma_post": sigma_post,
        "ci_low": float(mu_post - 1.96 * sigma_post),
        "ci_high": float(mu_post + 1.96 * sigma_post),
    }


# ════════════════════════════════════════════════════════════════════════
# FEATURE SCHEMA BUILDERS — kailash-ml FeatureSchema / FeatureField
# ════════════════════════════════════════════════════════════════════════


def build_schema_v1() -> Any:
    """Return the FeatureSchema v1 definition (basic property features)."""
    from kailash_ml.types import FeatureField, FeatureSchema

    return FeatureSchema(
        name="hdb_property_features",
        features=[
            FeatureField(
                name="floor_area_sqm",
                dtype="float64",
                nullable=False,
                description="Floor area in square metres",
            ),
            FeatureField(
                name="remaining_lease_years",
                dtype="float64",
                nullable=False,
                description="Remaining lease in years",
            ),
            FeatureField(
                name="storey_midpoint",
                dtype="float64",
                nullable=False,
                description="Midpoint of storey range",
            ),
            FeatureField(
                name="price_per_sqm",
                dtype="float64",
                nullable=False,
                description="Transaction price per square metre",
            ),
        ],
        entity_id_column="transaction_id",
        timestamp_column="transaction_date",
        version=1,
    )


def build_schema_v2() -> Any:
    """Return FeatureSchema v2 = v1 + three rolling market-context fields."""
    from kailash_ml.types import FeatureField, FeatureSchema

    v1 = build_schema_v1()
    return FeatureSchema(
        name="hdb_property_features",
        features=[
            *v1.features,
            FeatureField(
                name="town_median_price",
                dtype="float64",
                nullable=True,
                description="Median price in town (trailing 6 months)",
            ),
            FeatureField(
                name="town_transaction_volume",
                dtype="int64",
                nullable=True,
                description="Transaction count in town (trailing 6 months)",
            ),
            FeatureField(
                name="town_price_trend",
                dtype="float64",
                nullable=True,
                description="6-month price change % in town",
            ),
        ],
        entity_id_column="transaction_id",
        timestamp_column="transaction_date",
        version=2,
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 8.3: Rolling Features — Temporal Market Context
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Define FeatureSchema v2 with rolling market-context features
#   - Compute rolling statistics with Polars group_by_dynamic
#   - Understand rolling window warm-up periods and null handling
#   - Track schema evolution from v1 to v2 with versioned FeatureStore
#   - Apply rolling market features to Singapore town-level analytics
#
# PREREQUISITES: Exercise 8.1-8.2 (FeatureSchema v1, PIT retrieval)
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Theory — why rolling features capture market momentum
#   2. Build — define schema v2 and compute rolling town statistics
#   3. Train — register v2 and store in FeatureStore with versioning
#   4. Visualise — rolling price trends and transaction volumes by town
#   5. Apply — ERA Realty town-level investment advisory
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots



## THEORY — Why Rolling Features Capture Market Momentum


In [ ]:
# A single transaction price is a noisy signal. It depends on the
# specific flat's condition, the buyer's urgency, the agent's skill,
# and random timing. But the MEDIAN price in a town over the past 6
# months is a stable signal — it captures the local market's direction.
#
# Rolling features aggregate noisy individual observations into smooth
# market-level statistics:
#
#   - town_median_price: "What's the typical price in this town lately?"
#   - town_transaction_volume: "Is this a hot market or a cold one?"
#   - town_price_trend: "Are prices going up, down, or flat?"
#
# These three features transform a model from "predict based on this
# flat's characteristics" to "predict based on this flat in THIS market
# context". The same flat in a booming town sells for more than in a
# stagnant one.
#
# Polars group_by_dynamic is the engine: it buckets transactions into
# monthly windows per town, then rolling_mean/rolling_sum aggregates
# across a 6-month trailing window. The first 6 months per town have
# nulls — this is the warm-up period where the rolling window hasn't
# filled yet.
#
# Singapore context: HDB towns like Bishan, Tampines, and Woodlands
# have very different price trajectories. A 4-room flat in Bishan
# (mature estate, near MRT) appreciates differently from Woodlands
# (non-mature estate). Rolling features capture this divergence.



## TASK 2 — BUILD: Define FeatureSchema v2 and compute rolling features


In [ ]:
print("\n" + "=" * 70)
print("  Exercise 8.3 — Rolling Features: Temporal Market Context")
print("=" * 70)

# --- 2a. Load and compute v1 features (baseline) ---
hdb = load_hdb_resale()
features_v1 = compute_v1_features(hdb)

# --- 2b. Define FeatureSchema v2 ---
property_schema_v1 = build_schema_v1()

# TODO: Call the shared helper to build the v2 schema.
# Hint: build_schema_v2() returns a FeatureSchema that extends v1
# with three market-context fields: town_median_price,
# town_transaction_volume, town_price_trend.
property_schema_v2 = ____

n_new = len(property_schema_v2.features) - len(property_schema_v1.features)
print(f"\n  === FeatureSchema v2 (+{n_new} market features) ===")
for f in property_schema_v2.features:
    tag = (
        " [NEW]"
        if f.name not in [ff.name for ff in property_schema_v1.features]
        else ""
    )
    print(f"    {f.name}: {f.dtype} (nullable={f.nullable}){tag}")

# --- 2c. Compute v2 features ---
# TODO: Call compute_v2_features(hdb) to compute rolling market features.
# Hint: compute_v2_features adds town_median_price,
# town_transaction_volume, and town_price_trend via group_by_dynamic
# with a 6-month rolling window per town.
features_v2 = ____

# TODO: Count how many rows have non-null market context.
# Hint: filter for rows where town_median_price is not null, then
# get the .height (row count).
n_with_market = ____
pct_with_market = n_with_market / features_v2.height

print(f"\n  Computed v2 features: {features_v2.shape}")
print(f"  Rows with market context: {n_with_market:,} ({pct_with_market:.1%})")
print(f"  (First 6 months per town have nulls — rolling window warm-up)")

# --- 2d. Show sample rolling values for a few towns ---
sample_towns = ["ANG MO KIO", "BISHAN", "TAMPINES", "WOODLANDS"]
for town in sample_towns:
    town_data = features_v2.filter(
        (pl.col("town") == town) & pl.col("town_median_price").is_not_null()
    )
    if town_data.height > 0:
        latest = town_data.sort("transaction_date").tail(1)
        median_p = latest["town_median_price"].item()
        volume = latest["town_transaction_volume"].item()
        trend = latest["town_price_trend"].item()
        print(
            f"    {town:<15}: median=${median_p:>10,.0f}  "
            f"volume={volume:>5}  trend={trend:>+.1f}%"
        )



### Checkpoint 1


In [ ]:
assert (
    features_v2.height == features_v1.height
), "Task 2: v2 should have same row count as v1"
assert (
    "town_median_price" in features_v2.columns
), "Task 2: town_median_price must be computed"
assert (
    "town_transaction_volume" in features_v2.columns
), "Task 2: town_transaction_volume must be computed"
assert (
    "town_price_trend" in features_v2.columns
), "Task 2: town_price_trend must be computed"
assert (
    pct_with_market > 0.5
), f"Task 2: at least 50% of rows should have market context, got {pct_with_market:.1%}"
print("\n[ok] Checkpoint 1 passed — v2 features computed with rolling market context\n")

# INTERPRETATION: The v2 schema adds three nullable columns. They're
# nullable because the first 6 months per town can't compute a
# trailing window — that's the warm-up period, not a data quality bug.
# Downstream models must drop_nulls on these columns before training.



## TASK 3 — TRAIN: Register v2 schema and store versioned features


In [ ]:
print("--- FeatureStore Schema Evolution (v1 -> v2) ---")

factory, fs, tracker, has_backend  = await setup_feature_store()

if has_backend:
    try:

        async def store_v2():
            await fs.register_features(property_schema_v2)
            return await fs.store(features_v2, property_schema_v2)

        row_count  = await store_v2()
        print(f"  Stored {row_count:,} v2 feature rows")
        print(f"  Schema version: {property_schema_v2.version}")
    except Exception as e:
        has_backend = False
        print(f"  [Skipped: v2 store ({type(e).__name__}: {e})]")
else:
    print("  [Skipped: FeatureStore backend unavailable]")
    print("  v2 features remain in-memory as Polars DataFrame")

print(f"\n  Schema evolution:")
print(f"    v1: {len(property_schema_v1.features)} features (basic property)")
print(f"    v2: {len(property_schema_v2.features)} features (+{n_new} market context)")
print(f"    New fields: town_median_price, town_transaction_volume, town_price_trend")



### Checkpoint 2


In [ ]:
assert property_schema_v2.version == 2, "Task 3: v2 schema must be version 2"
assert len(property_schema_v2.features) == 7, "Task 3: v2 must have 7 features"
print("\n[ok] Checkpoint 2 passed — v2 schema registered and stored\n")



## TASK 4 — VISUALISE: Rolling price trends and volumes by town


In [ ]:
print("--- Town-Level Rolling Market Trends ---")

# Aggregate monthly medians per town for plotting
town_monthly = (
    features_v2.filter(pl.col("town_median_price").is_not_null())
    .group_by(["town", "transaction_date"])
    .agg(
        pl.col("town_median_price").first(),
        pl.col("town_transaction_volume").first(),
        pl.col("town_price_trend").first(),
    )
    .sort("transaction_date")
)

# TODO: Create a 2-row subplot figure showing rolling median price
# (top) and transaction volume (bottom) for the sample towns.
# Hint: make_subplots(rows=2, cols=1, ...) then add go.Scatter traces
# for each town in sample_towns.
fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=[
        "Rolling 6-Month Median Price by Town",
        "Rolling 6-Month Transaction Volume by Town",
    ],
    vertical_spacing=0.12,
)

for town in sample_towns:
    town_data = town_monthly.filter(pl.col("town") == town).sort("transaction_date")
    if town_data.height > 0:
        dates = town_data["transaction_date"].to_list()
        fig.add_trace(
            go.Scatter(
                x=dates,
                y=town_data["town_median_price"].to_list(),
                name=town,
                mode="lines",
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=dates,
                y=town_data["town_transaction_volume"].to_list(),
                name=town,
                showlegend=False,
                mode="lines",
            ),
            row=2,
            col=1,
        )

fig.update_layout(
    title="HDB Market Trends by Town (Rolling 6-Month Window)",
    height=700,
    width=1000,
)
fig.update_yaxes(title_text="Median Price ($)", row=1, col=1)
fig.update_yaxes(title_text="Transaction Count", row=2, col=1)

fig.write_html(str(OUTPUT_DIR / "03_town_trends.html"))
print(f"\n  Saved: {OUTPUT_DIR / '03_town_trends.html'}")

# TODO: Create a bar chart of average price trend by town.
# Hint: group features_v2 by town, compute mean of town_price_trend,
# sort by avg_trend, and use go.Bar with colour conditional on sign.
trend_data = (
    features_v2.filter(pl.col("town_price_trend").is_not_null())
    .group_by("town")
    .agg(pl.col("town_price_trend").mean().alias("avg_trend"))
)

fig2 = go.Figure()
fig2.add_trace(
    go.Bar(
        x=trend_data.sort("avg_trend")["town"].to_list(),
        y=trend_data.sort("avg_trend")["avg_trend"].to_list(),
        marker_color=[
            "firebrick" if t < 0 else "seagreen"
            for t in trend_data.sort("avg_trend")["avg_trend"].to_list()
        ],
    )
)
fig2.update_layout(
    title="Average 6-Month Price Trend by Town (%)",
    xaxis_title="Town",
    yaxis_title="Average Price Trend (%)",
    xaxis_tickangle=-45,
)
fig2.write_html(str(OUTPUT_DIR / "03_town_price_trends.html"))
print(f"  Saved: {OUTPUT_DIR / '03_town_price_trends.html'}")



### Checkpoint 3


In [ ]:
print("\n[ok] Checkpoint 3 passed — rolling market trends visualised\n")



## TASK 5 — APPLY: ERA Realty Town-Level Investment Advisory


In [ ]:
# Scenario: ERA Realty advisors help HDB upgraders decide WHEN and
# WHERE to buy. With rolling market features, advisors can identify
# towns where prices are trending up (buy soon) vs towns where prices
# are stagnant (negotiate harder).
#
# Without rolling features: advisors rely on gut feel and last month's
# newspaper headlines. "Bishan is always expensive" — but IS it still
# appreciating, or has it plateaued?
#
# With rolling features: advisors see quantified 6-month trends per
# town. "Bishan median up 3.2% vs Tampines up 7.1% — Tampines is
# gaining ground. Buy Tampines now before the gap closes."
#
# ERA has ~6,800 agents in Singapore. If rolling-feature-based advice
# helps each agent close 1 additional deal per quarter (conservative),
# at an average commission of $5,000:
#   6,800 agents * 1 deal/quarter * $5,000 = S$34M additional revenue/year

print("=== APPLY: ERA Realty Town-Level Investment Advisory ===")

# TODO: Rank towns by their latest price trend (descending).
# Hint: group features_v2 by town, take .last() of each rolling
# column, then sort by latest_trend descending.
latest_trends = (
    features_v2.filter(pl.col("town_price_trend").is_not_null())
    .group_by("town")
    .agg(
        pl.col("town_price_trend").last().alias("latest_trend"),
        pl.col("town_median_price").last().alias("latest_median"),
        pl.col("town_transaction_volume").last().alias("latest_volume"),
    )
    .sort("latest_trend", descending=True)
)

print()
print("  Top 5 appreciating towns (buy-soon signal):")
for row in latest_trends.head(5).iter_rows(named=True):
    print(
        f"    {row['town']:<15}: trend={row['latest_trend']:>+6.1f}%  "
        f"median=${row['latest_median']:>10,.0f}  volume={row['latest_volume']:>5}"
    )

print()
print("  Bottom 5 towns (negotiate-harder signal):")
for row in latest_trends.tail(5).iter_rows(named=True):
    print(
        f"    {row['town']:<15}: trend={row['latest_trend']:>+6.1f}%  "
        f"median=${row['latest_median']:>10,.0f}  volume={row['latest_volume']:>5}"
    )

print()
print("  ERA advisory impact:")
print("    - 6,800 agents with quantified town-level trends")
print("    - 1 additional deal/quarter per agent (conservative)")
print("    - S$34M additional revenue per year")
print()
print("  Key insight: 'Bishan is expensive' is qualitative.")
print("  'Bishan median up 3.2% vs Tampines up 7.1% over 6 months'")
print("  is quantitative and actionable.")



### Checkpoint 4


In [ ]:
assert latest_trends.height > 0, "Task 5: must have town trend data"
print("\n[ok] Checkpoint 4 passed — town-level advisory demonstrated\n")



## REFLECTION


[ok] FeatureSchema v2: extending v1 with rolling market-context fields
  [ok] group_by_dynamic: monthly bucketing of transactions per town
  [ok] Rolling statistics: trailing 6-month median, volume, trend
  [ok] Warm-up periods: why the first 6 months per town have nulls
  [ok] Schema versioning: v1 -> v2 evolution with backward compatibility

  KEY INSIGHT: Rolling features transform a model from "what is this
  flat worth?" to "what is this flat worth IN THIS MARKET?" The same
  flat in a booming town sells for more than in a stagnant one — and
  rolling features capture that difference quantitatively.

  Next: In 04_modeling_lineage.py, you'll build a full regression model
  on v2 features, apply hypothesis tests and Bayesian posteriors to the
  coefficients, and create a complete audit trail from data to model.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

